In [2]:
import os

import ee
import geemap.foliumap as geemap
import geemap

from find_set_root import find_set_project_root
PROJECT_ROOT = find_set_project_root()
print(f"Project root found at: {PROJECT_ROOT}")
import utils.general_functions as ugf


DIR_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
DIR_DERIVED = os.path.join(PROJECT_ROOT, 'data', 'derived')
ugf.dir_ensure([DIR_RAW, DIR_DERIVED])

#Prepare to use Earth Engine
ee.Authenticate()
ee.Initialize(project = 'ee-tymc5571-multi-disturbance')

cbi = ee.Image("projects/ee-tymc5571-goodfire/assets/welty_wildfire_cbi_bc_1985_2020")
lcms = ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
lcmap = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI")

#Add CBI prefix if not present
cbi = cbi.rename(
    cbi.bandNames().map(
        lambda n: ee.Algorithms.If(
            ee.String(n).slice(0, 4).compareTo('cbi_').eq(0),
            n,
            ee.String('cbi_').cat(n)
        )
    )
)

# The temporal AOI will be based on the years in the CBI data selected
years = cbi.bandNames().map(lambda b: ee.Number.parse(ee.String(b).split('_').get(2)))
print(years.getInfo())
CBI_START_YEAR = ee.Number(years.sort().get(0)).getInfo()
CBI_END_YEAR = ee.Number(years.sort().get(-1)).getInfo()
print(CBI_START_YEAR, CBI_END_YEAR)


c:\Users\tymc5571\AppData\Local\miniconda3\envs\macrosystems\lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Project root found at: C:\Users\tymc5571\dev\compound-disturbance-resilience
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\raw
✅ Directory already exists: C:\Users\tymc5571\dev\compound-disturbance-resilience\data\derived
[1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
1985 2020


In [3]:
# FUNCTIONS


def reclassify_image_binary(value):
    def _reclassify(img):
        binary_img = img.eq(value).selfMask()
        return binary_img.copyProperties(img, ['system:time_start'])
    return _reclassify

# -------------------------------
# Combine two collections using logical AND (per image pair), copying time from the first image
def combine_collections_with_and(collection1, collection2, and_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    combined = list1.zip(list2)

    def combine_pair(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        and_result = image1.And(image2).rename(and_name)
        return and_result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(combined.map(combine_pair))

# -------------------------------
# Add a 'year' property to each image
def add_year_property(image):
    year = image.date().get('year')
    return image.set('year', year)

def toBands_with_projection(collection):
    collection = ee.ImageCollection(collection)
    image = collection.toBands()
    reference_img = ee.Image(collection.first())
    return image.setDefaultProjection(reference_img.projection())


def remove_to_bands_append(image):
    """
    Renames bands in the input Earth Engine Image by removing the first part
    (before the first underscore) from each band name.

    Args:
        image (ee.Image): The input image.

    Returns:
        ee.Image: The image with renamed bands.
    """
    old_names = image.bandNames()
    new_names = old_names.map(lambda name: ee.String(name).split('_').slice(1).join('_'))
    return image.rename(new_names)



def insert_to_band_names(image: ee.Image, text: str, index: int) -> ee.Image:
    """
    Inserts a string into each band name of an Earth Engine image at the specified index.
    
    Args:
        image (ee.Image): The input image.
        text (str): The string to insert.
        index (int): The character index at which to insert the string.
                     Supports negative indexing from the end.
    
    Returns:
        ee.Image: Image with updated band names.
    """
    def insert_at(name):
        name_str = ee.String(name)
        name_len = name_str.length()
        
        # Compute final insertion index (accounting for negative values)
        insert_pos = ee.Algorithms.If(
            index >= 0,
            ee.Number(index),
            name_len.add(index)  # index is negative, so subtract from length
        )
        insert_pos = ee.Number(insert_pos).clamp(0, name_len)  # avoid out-of-bounds
        
        prefix = name_str.slice(0, insert_pos)
        suffix = name_str.slice(insert_pos)
        return prefix.cat(text).cat(suffix)

    old_names = image.bandNames()
    new_names = old_names.map(insert_at)
    return image.rename(new_names)


In [20]:
###################
# Relaxed forest mask
###################

lcmap_cover = lcmap.filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
lcms_cover = lcms.filter(ee.Filter.eq('study_area', 'CONUS')).filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR)).select('Land_Cover')

lcmap_for = lcmap_cover.map(reclassify_image_binary(4))
lcms_for = lcms_cover.map(reclassify_image_binary(1))

combined_forest_mask = lcmap_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max().Or(
    lcms_for.map(lambda img: img.reduce(ee.Reducer.max()).gt(0)).max()
)
combined_forest_mask = combined_forest_mask.setDefaultProjection(lcmap_cover.first().projection())

###################
# Conservative forest by year
###################

# Filter lcmap and lcms to relevant years and region
lcmap_cover_conus = lcmap.filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
lcms_cover_conus = (
    lcms
    .filter(ee.Filter.eq('study_area', 'CONUS'))
    .filterDate(str(CBI_START_YEAR - 1), str(CBI_END_YEAR))
    .select('Land_Cover')
)

# Reclassify to forest (value = 4 for lcmap, value = 1 for lcms)
lcmap_for = lcmap_cover_conus.map(reclassify_image_binary(4))
lcms_for = lcms_cover_conus.map(reclassify_image_binary(1))

# Combine both masks conservatively (AND)
conservative_forest_by_year = combine_collections_with_and(lcmap_for, lcms_for, 'conservativeForest').map(add_year_property)

###################
# Create distance from unburned forest
###################


print(conservative_forest_by_year.filter(ee.Filter.eq('year', years.get(0))).first().bandNames().getInfo())

# Get a list of all 'year' properties in conservative_forest
years_in_conservative_forest = conservative_forest_by_year.aggregate_array('year').getInfo()
print("Years in conservative_forest:", years_in_conservative_forest)

def unburned_forest_mask(year):
    band_name = ee.String('cbi_year_').cat(ee.String(year))
    cbi_year = cbi.select([band_name])
    forest_img = conservative_forest_by_year.filter(ee.Filter.eq('year', ee.Number(year).subtract(1))).first()
    return ee.Algorithms.If(
        forest_img,
        (lambda:
            (cbi_year.lt(2.25).unmask(0).Or(cbi_year.mask().Not()))
            .And(ee.Image(forest_img).mask())
            .selfMask()
            #.rename(ee.String('year_').cat(ee.String(year)).cat('_unburned_forest'))
            .rename(ee.String(year))
            .set('year', year)
        )(),
        None  # Skip year if forest_img is None; note that this will remove 1985, as the first year of forest cover data is 1985
    )

unburned_forest_images = years.map(unburned_forest_mask)
# Convert to ImageCollection
unburned_forest = ee.ImageCollection(unburned_forest_images)#.toBands()
unburned_forest = toBands_with_projection(unburned_forest)
unburned_forest = remove_to_bands_append(unburned_forest)
unburned_forest = insert_to_band_names(image=unburned_forest, text='unburned_forest_', index=0)


print(unburned_forest.bandNames().getInfo())

# GET DISTANCE
band_names = unburned_forest.bandNames()

# Define the function to apply to each band
def get_unburned_distance(band_name):
    band_name = ee.String(band_name)
    band = unburned_forest.select([band_name])
    transformed = band.fastDistanceTransform(30).sqrt().multiply(30)
    #return transformed.rename(band_name.cat('_distance'))
    return transformed.rename(band_name)

# Map over the band names and create an ImageCollection
transformed_bands = ee.ImageCollection(band_names.map(get_unburned_distance))

# Convert the ImageCollection to a single Image
unburned_forest_distance = toBands_with_projection(transformed_bands)#.toBands()
unburned_forest_distance = remove_to_bands_append(unburned_forest_distance)
unburned_forest_distance = insert_to_band_names(image=unburned_forest_distance, text='distance_', index=0)


print(unburned_forest_distance.bandNames().getInfo())

print('CBI:', cbi.bandNames().getInfo())

['conservativeForest']
Years in conservative_forest: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
['unburned_forest_1986', 'unburned_forest_1987', 'unburned_forest_1988', 'unburned_forest_1989', 'unburned_forest_1990', 'unburned_forest_1991', 'unburned_forest_1992', 'unburned_forest_1993', 'unburned_forest_1994', 'unburned_forest_1995', 'unburned_forest_1996', 'unburned_forest_1997', 'unburned_forest_1998', 'unburned_forest_1999', 'unburned_forest_2000', 'unburned_forest_2001', 'unburned_forest_2002', 'unburned_forest_2003', 'unburned_forest_2004', 'unburned_forest_2005', 'unburned_forest_2006', 'unburned_forest_2007', 'unburned_forest_2008', 'unburned_forest_2009', 'unburned_forest_2010', 'unburned_forest_2011', 'unburned_forest_2012', 'unburned_forest_2013', 'unburned_forest_2014', 'unburned_forest_2015', 'unburned_fores

In [21]:
Map = geemap.Map(center=[40.0, -105.5], zoom=11)

Map.addLayer(combined_forest_mask.updateMask(combined_forest_mask), {
    'palette': ['lightgreen']
}, 'Combined Forest Mask')

Map.addLayer(conservative_forest_by_year.filter(ee.Filter.eq('year', 1985)), {
    'palette': ['darkgreen']
}, 'Conservative Forest 1985')

Map.addLayer(conservative_forest_by_year.filter(ee.Filter.eq('year', 1986)), {
    'palette': ['darkgreen']
}, 'Conservative Forest 1986')

Map.addLayer(conservative_forest_by_year.filter(ee.Filter.eq('year', 1987)), {
    'palette': ['purple']
}, 'Conservative Forest 1987')



Map.addLayer(cbi.select('cbi_year_1986'), {
    'palette': ['blue', 'green', 'yellow', 'orange', 'red'],
    'min': 0, 'max': 3
}, 'CBI 1986')


Map.addLayer(cbi.select('cbi_year_1985'), {
    'palette': ['blue', 'green', 'yellow', 'orange', 'red'],
    'min': 0, 'max': 3
}, 'CBI 1985')

Map.addLayer(cbi.select('cbi_year_1986'), {
    'palette': ['blue', 'green', 'yellow', 'orange', 'red'],
    'min': 0, 'max': 3
}, 'CBI 1986')


Map.addLayer(unburned_forest.select('unburned_forest_1986'), {
    'palette': ['green']
}, 'Unburned Forest 1986')

Map.addLayer(unburned_forest_distance.select('distance_unburned_forest_1986'), {
    'min': 0,
    'max': 1000,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}, 'Unburned Forest Distance 1986')

Map.addLayer(unburned_forest_distance.select('distance_unburned_forest_1987'), {
    'min': 0,
    'max': 1000,
    'palette': ['blue', 'green', 'yellow', 'orange', 'red']
}, 'Unburned Forest Distance 1987')

Map

Map(center=[40.0, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…